# 📘 Session 6 — LangGraph: State, Reducers, Memory & Cross-Thread Stores

**What you will learn:**
| Topic | Key Idea |
|-------|----------|
| **State & Reducers** | How `Annotated[list, operator.add]` accumulates values instead of overwriting them |
| **Short-term Memory** | `InMemorySaver` checkpointer keeps per-thread conversation history |
| **Cross-thread Store** | `InMemoryStore` lets you persist facts *across* different threads / sessions |
| **Custom Agent Loop** | Build a tool-calling ReAct agent from scratch with `StateGraph` |
| **LangSmith Tracing** | Every graph run is automatically traced in LangSmith |

## 1. Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("LangSmith Tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith Project:", os.getenv("LANGSMITH_PROJECT"))

## 2. State & Reducers — The Core Concept

In a LangGraph graph every node reads **and writes** to a shared `State` dict.
By default a node's output *overwrites* the previous value.

> **Reducer = a function that decides how to merge two values.**
> `Annotated[list[str], operator.add]` means: *append* new items instead of overwriting.

| Field | Type | Behaviour |
|-------|------|-----------|
| `foo` | `str` | Overwrite — latest node wins |
| `car` | `list[str]` | Overwrite — latest node wins |
| `bar` | `Annotated[list[str], add]` | Accumulate — values are concatenated |
| `name` | `Annotated[list[str], add]` | Accumulate — values are concatenated |

In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]   # reducer: append
    car: list[str]                    # no reducer: overwrite
    name: Annotated[list[str], add]  # reducer: append — MUST return list, not str!

def node_a(state: State):
    return {
        "foo": "a",
        "bar": ["a"],
        "car": [],
        "name": ["sourav"]  # list! operator.add expects list + list
    }

def node_b(state: State):
    return {
        "foo": "b",
        "bar": ["b"],
        "car": ["mercedes"],
        "name": ["abhi"]    # list!
    }

workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

graph = workflow.compile(checkpointer=InMemorySaver())

config: RunnableConfig = {"configurable": {"thread_id": "demo-1"}}
result = graph.invoke({"foo": "", "bar": [], "car": [], "name": []}, config)

print("Final State:")
for k, v in result.items():
    print(f"  {k}: {v}")

# Expected:
# foo: b          <- overwritten by node_b
# bar: ['a','b']  <- accumulated by both nodes
# car: ['mercedes']  <- overwritten by node_b
# name: ['sourav','abhi']  <- accumulated by both nodes

## 3. Short-Term Memory — `InMemorySaver` (Checkpointer)

A **checkpointer** saves the full graph state after every step.
When you send a second message on the **same `thread_id`**, LangGraph reloads the
saved snapshot so the conversation continues where it left off.

```
Thread "A"  → its own history (private)
Thread "B"  → its own history (private)
```

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

# A simple mock LLM that always replies with the same message
def assistant(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "Hi! I am Sourav's research assistant. How can I help?"}]}

graph2 = StateGraph(MessagesState)
graph2.add_node(assistant)
graph2.add_edge(START, "assistant")
graph2.add_edge("assistant", END)
graph2 = graph2.compile(checkpointer=InMemorySaver())

thread = {"configurable": {"thread_id": "chat-1"}}

# First turn
r1 = graph2.invoke({"messages": [{"role": "user", "content": "Hello, my name is Sourav."}]}, thread)
print("Turn 1 messages:", len(r1["messages"]))

# Second turn — conversation grows because checkpointer restored the history
r2 = graph2.invoke({"messages": [{"role": "user", "content": "Who am I?"}]}, thread)
print("Turn 2 messages:", len(r2["messages"]))
print("\nFull conversation:")
for m in r2["messages"]:
    role = getattr(m, "type", "?")
    print(f"  [{role}]: {m.content}")

## 4. Cross-Thread Memory — `InMemoryStore`

The checkpointer only remembers things *within* one thread.
If the user opens a **brand-new conversation**, that memory is gone.

`InMemoryStore` is a **namespace-based key-value store** that lives *outside* any thread,
so facts stored here are available across all sessions.

```
namespace = (user_id, "memories")
store.put(namespace, key, value_dict)
store.search(namespace)  → list of items
store.get(namespace, key)  → single item
```

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

user_id = "user-sourav"
namespace = (user_id, "memories")

# Store a preference
memory_id = str(uuid.uuid4())  # unique key for this memory item
store.put(namespace, memory_id, {"food_preference": "I like pizza"})

# Retrieve it later (even from a different thread)
memories = store.search(namespace)
if memories:
    print("Latest memory:", memories[-1].dict())
else:
    print("No memories found.")

In [ ]:
# Store multiple memories and retrieve them all
store.put(namespace, str(uuid.uuid4()), {"hobby": "playing football"})
store.put(namespace, str(uuid.uuid4()), {"city": "Dhaka"})

all_memories = store.search(namespace)
print(f"Total stored memories: {len(all_memories)}")
for item in all_memories:
    print(" -", item.value)

## 5. Using the Store *Inside* a Graph Node

Nodes can receive the store via dependency injection — just declare a `store` parameter
and LangGraph passes it automatically when you compile with `store=...`.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

long_term_store = InMemoryStore()

# Pre-seed the store with some facts about the user
long_term_store.put(("user-1", "profile"), "pref-1", {"fact": "Likes Python and AI"})
long_term_store.put(("user-1", "profile"), "pref-2", {"fact": "Lives in Dhaka"})

def personalised_assistant(state: MessagesState, store: InMemoryStore):
    # Read cross-thread memories for this user
    memories = store.search(("user-1", "profile"))
    facts = ", ".join(m.value.get("fact", "") for m in memories)

    reply = f"[I know about you: {facts}] — How can I help you today?"
    return {"messages": [{"role": "ai", "content": reply}]}

g = StateGraph(MessagesState)
g.add_node(personalised_assistant)
g.add_edge(START, "personalised_assistant")
g.add_edge("personalised_assistant", END)
g = g.compile(checkpointer=InMemorySaver(), store=long_term_store)

res = g.invoke(
    {"messages": [{"role": "user", "content": "Hi!"}]},
    {"configurable": {"thread_id": "new-session-999"}}
)
print(res["messages"][-1].content)

## 6. Building a Custom Tool-Calling Agent with LangGraph

Instead of using `create_agent`, we can build our own ReAct loop from scratch.
This is powerful because we control every step.

```
START
  └─► llm_call ─────────────────────────────┐
          │ tool_calls present?               │
         yes ──► tool_node ──► llm_call       │
          │ no                                │
         END ◄──────────────────────────────-┘
```

In [ ]:
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, ToolMessage, AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated, Literal
import operator

# --- Tools ---
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

@tool
def add_numbers(a: int, b: int) -> int:
    """Add a and b."""
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divide a by b."""
    return a / b

tools = [multiply, add_numbers, divide]
tools_by_name = {t.name: t for t in tools}

# --- Model ---
model = ChatOllama(model="llama3.2:3b", temperature=0)
model_with_tools = model.bind_tools(tools)

print("Tools registered:", list(tools_by_name.keys()))

In [ ]:
# --- Agent State ---
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# --- Nodes ---
def llm_call(state: AgentState):
    """Ask the LLM — it will decide whether to call a tool or reply."""
    system = SystemMessage(content="You are a helpful math assistant. Use tools to answer accurately.")
    response = model_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState):
    """Run all tool calls requested by the LLM."""
    results = []
    for call in state["messages"][-1].tool_calls:
        tool = tools_by_name[call["name"]]
        output = tool.invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

def should_continue(state: AgentState) -> Literal["tool_node", "__end__"]:
    """Route: if the LLM made tool calls → run tools; otherwise → finish."""
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return "__end__"

# --- Build Graph ---
agent_builder = StateGraph(AgentState)
agent_builder.add_node(llm_call)
agent_builder.add_node(tool_node)
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
agent_builder.add_edge("tool_node", "llm_call")

agent = agent_builder.compile(checkpointer=InMemorySaver())
print("Custom agent compiled successfully.")

In [ ]:
from langchain_core.messages import HumanMessage

# Ask the agent a math question
result = agent.invoke(
    {"messages": [HumanMessage(content="What is 6 multiplied by 7, then add 5?")]},
    {"configurable": {"thread_id": "math-1"}}
)

print("Agent answer:", result["messages"][-1].content)

## 7. Pagination & Bulk Store Operations

When you have many items in the store you can paginate using `limit` and `offset`.

In [ ]:
bulk_store = InMemoryStore()
ns = ("demo", "items")

# Insert 10 items
for i in range(10):
    bulk_store.put(ns, str(uuid.uuid4()), {"index": i, "value": f"item-{i}"})

# Paginate: read 4 at a time
page_size = 4
offset = 0
total_seen = 0

while True:
    page = bulk_store.search(ns, limit=page_size, offset=offset)
    if not page:
        break
    total_seen += len(page)
    print(f"Page offset={offset}: {[item.value['value'] for item in page]}")
    offset += page_size

print(f"Total items retrieved: {total_seen}")

## 8. Key Takeaways

| Concept | When to use |
|---------|-------------|
| `Annotated[list, add]` reducer | Accumulate messages or facts across multiple nodes |
| `InMemorySaver` checkpointer | Persist conversation *within* one session / thread |
| `InMemoryStore` | Share facts *across* sessions / different users |
| Custom agent loop | Full control over how tools are called and when to stop |
| LangSmith tracing | Automatic — just set `LANGSMITH_TRACING=true` in `.env` |

> **Next:** Session 7 will add a real LLM to the custom agent loop and wire it into the Northstar support system.